# V9 CIFAR-10 Feature Extraction T4x2 1k Hardened

`NO_FAKE_RESULTS`  
`NO_REAL_EVIDENCE`  
`not paper evidence`  
`claim_allowed=false`

This notebook stops after feature-cache packaging. It does not run metric reproduction or certificates.

In [ ]:
import concurrent.futures, hashlib, json, os, shutil, stat, subprocess, sys, time, zipfile
from pathlib import Path
import numpy as np, torch
print('python', sys.version)
print('gpu_count', torch.cuda.device_count())
print('gpu_names', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('disk', shutil.disk_usage('/kaggle/working'))
if torch.cuda.device_count() < 2:
    Path('/kaggle/working/feature_blocked_status.json').write_text(json.dumps({'status_code':'BLOCKED_CUDA_T4X2_NOT_AVAILABLE','claim_allowed':False}, indent=2))
    raise RuntimeError('T4x2 required')

In [ ]:
!pip -q install 'transformers==4.53.2' 'timm==1.0.16' 'pillow==11.2.1' 'scipy==1.15.3' 'PyYAML==6.0.2'
!python -m pip freeze > /kaggle/working/feature_dependency_freeze.txt

In [ ]:
INPUT_ZIP=Path('/kaggle/input/certgen-features/certgen_cifar10_feature_extraction_1k_input.zip')
WORK=Path('/kaggle/working/v9_feature_input')
if not INPUT_ZIP.exists():
    Path('/kaggle/working/feature_blocked_status.json').write_text(json.dumps({'status_code':'BLOCKED_FEATURE_INPUT_ZIP_MISSING','claim_allowed':False}, indent=2))
    raise FileNotFoundError(INPUT_ZIP)
input_hash=hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
marker=WORK/'.source_zip_sha256'
if WORK.exists():
    if not marker.is_file() or marker.read_text().strip()!=input_hash: raise RuntimeError('existing input extraction does not match attached ZIP')
else:
    WORK.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(INPUT_ZIP) as archive:
        infos=archive.infolist(); total=sum(i.file_size for i in infos)
        if len(infos)>100000 or total>20*1024**3 or archive.testzip() is not None: raise RuntimeError('input ZIP failed size/CRC limits')
        seen=set()
        for info in infos:
            parts=Path(info.filename).parts; mode=(info.external_attr>>16)&0xFFFF
            if info.filename.startswith('/') or '..' in parts or '\\' in info.filename or info.filename.casefold() in seen or (mode and stat.S_ISLNK(mode)): raise RuntimeError(f'unsafe input ZIP member: {info.filename}')
            seen.add(info.filename.casefold())
            target=WORK/info.filename
            if info.is_dir(): target.mkdir(parents=True, exist_ok=True); continue
            target.parent.mkdir(parents=True, exist_ok=True); target.write_bytes(archive.read(info))
    marker.write_text(input_hash)
sys.path.insert(0,str(WORK/'repo')); os.environ['PYTHONPATH']=str(WORK/'repo')
sample_manifest=WORK/'manifests/cifar10_r1_feature_extraction_samples.jsonl'
config=json.loads((WORK/'config/feature_extraction_config.json').read_text())
assert config['claim_allowed'] is False
expected_extractor_locks={'inception_v3_pool3':'torchvision::Inception_V3_Weights.IMAGENET1K_V1','clip_vit':'openai/clip-vit-large-patch14@32bd64288804d66eefd0ccbe215aa642df71cc41'}
assert config.get('extractor_locks')==expected_extractor_locks
assert '32bd64288804d66eefd0ccbe215aa642df71cc41' in (WORK/'repo/certgen/features/extractors/clip.py').read_text()
expected_roles={'reference','google_ddpm','frank_ddpm_ema','frank_cfm'}
roles=[]
for line in sample_manifest.read_text().splitlines():
    if line.strip(): roles.append(json.loads(line)['role'])
assert expected_roles <= set(roles), set(roles)
lock=list((WORK/'configs').glob('*.json'))[0]
lock_hash=hashlib.sha256(lock.read_bytes()).hexdigest()
print({'preprocessing_lock':str(lock),'preprocessing_lock_hash':lock_hash})

In [ ]:
FEATURE_ROOT=Path('/kaggle/working/features/cifar10_r1')
for p in [FEATURE_ROOT/'inception', FEATURE_ROOT/'clip', FEATURE_ROOT/'split', FEATURE_ROOT/'logs', FEATURE_ROOT/'status']:
    p.mkdir(parents=True, exist_ok=True)
def shard_cache_valid(extractor, subdir, gpu):
    shard=FEATURE_ROOT/subdir/f'shard-{gpu:03d}-of-002'; npz=shard/f'{extractor}_features.npz'; sidecar=shard/f'{extractor}_features.json'
    if not npz.is_file() or not sidecar.is_file(): return False
    try:
        meta=json.loads(sidecar.read_text())
        with np.load(npz,allow_pickle=False) as loaded: arr=np.asarray(loaded['features']); ids=[str(x) for x in loaded['sample_ids']] if 'sample_ids' in loaded else meta.get('sample_ids',[])
        declared=meta.get('features_sha256') or meta.get('hash') or (meta.get('hashes') or {}).get('features_sha256')
        return arr.ndim==2 and len(ids)==arr.shape[0] and len(ids)==len(set(ids)) and np.isfinite(arr).all() and declared==hashlib.sha256(npz.read_bytes()).hexdigest() and meta.get('source_manifest_sha256')==hashlib.sha256(sample_manifest.read_bytes()).hexdigest() and meta.get('preprocessing_lock_sha256')==lock_hash and meta.get('claim_allowed') is False
    except Exception: return False
def run_extractor_shard(extractor, subdir, batch_size, gpu):
    status_path=FEATURE_ROOT/'status'/f'{extractor}_gpu{gpu}_status.json'
    if status_path.exists() and json.loads(status_path.read_text()).get('status_code')=='EXTRACTOR_SHARD_COMPLETE' and shard_cache_valid(extractor,subdir,gpu): return json.loads(status_path.read_text())
    args=[sys.executable,'-m','certgen.features.extract','--input-manifest',str(sample_manifest),'--provenance-ledger',str(WORK/'inputs/cifar10_r1_ledger.csv'),'--preprocessing-lock',str(lock),'--extractor',extractor,'--out-dir',str(FEATURE_ROOT/subdir),'--device','cuda','--batch-size',str(batch_size),'--shard-id',str(gpu),'--num-shards','2','--resume','--execute']
    cmd=f"CUDA_VISIBLE_DEVICES={gpu} "+' '.join(args); start=time.time(); env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu))
    with (FEATURE_ROOT/'logs'/f'{extractor}_gpu{gpu}.log').open('a') as log: code=subprocess.run(args,env=env,stdout=log,stderr=subprocess.STDOUT).returncode
    complete=code==0 and shard_cache_valid(extractor,subdir,gpu)
    payload={'extractor':extractor,'gpu':gpu,'status_code':'EXTRACTOR_SHARD_COMPLETE' if complete else 'EXTRACTOR_SHARD_FAILED','wall_time_seconds':time.time()-start,'resume_supported':True,'failed_shard_rerun':cmd,'claim_allowed':False}
    temporary=status_path.with_suffix('.json.tmp'); temporary.write_text(json.dumps(payload,indent=2)); os.replace(temporary,status_path); return payload
def run_extractor(extractor, subdir, batch_size):
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
        statuses=[future.result() for future in [pool.submit(run_extractor_shard,extractor,subdir,batch_size,0),pool.submit(run_extractor_shard,extractor,subdir,batch_size,1)]]
    if any(s['status_code']!='EXTRACTOR_SHARD_COMPLETE' for s in statuses):
        Path('/kaggle/working/feature_blocked_status.json').write_text(json.dumps({'status_code':'BLOCKED_FEATURE_SHARD_FAILED','extractor':extractor,'statuses':statuses,'claim_allowed':False}, indent=2))
        raise RuntimeError('feature shard failed')
    return statuses
all_status=run_extractor('inception_v3_pool3','inception',64)+run_extractor('clip_vit','clip',64)
(FEATURE_ROOT/'status'/'feature_extraction_status.json').write_text(json.dumps({'status_code':'FEATURE_EXTRACTION_SHARDS_COMPLETE','extractor_statuses':all_status,'evidence_status':'run_log_only','claim_allowed':False}, indent=2))

In [ ]:
merge_specs=[('inception','inception_v3_pool3'),('clip','clip_vit')]
for subdir,extractor in merge_specs:
    subprocess.run([sys.executable,'-m','certgen.features.merge_shards','--shard-dir',str(FEATURE_ROOT/subdir/'shard-000-of-002'),'--shard-dir',str(FEATURE_ROOT/subdir/'shard-001-of-002'),'--extractor',extractor,'--out-npz',str(FEATURE_ROOT/f'cifar10_r1_{subdir}.npz'),'--out-sidecar',str(FEATURE_ROOT/f'cifar10_r1_{subdir}.sidecar.json')],check=True)
    subprocess.run([sys.executable,'-m','certgen.features.split_by_role','--features-npz',str(FEATURE_ROOT/f'cifar10_r1_{subdir}.npz'),'--sidecar',str(FEATURE_ROOT/f'cifar10_r1_{subdir}.sidecar.json'),'--sample-manifest',str(sample_manifest),'--extractor-label',subdir,'--out-dir',str(FEATURE_ROOT/'split'),'--summary-out',str(FEATURE_ROOT/'status'/f'{subdir}_split_summary.json')],check=True)

In [ ]:
required=['reference_inception','google_ddpm_inception','frank_ddpm_ema_inception','frank_cfm_inception','reference_clip','google_ddpm_clip','frank_ddpm_ema_clip','frank_cfm_clip']
role_status=[]
for item in required:
    npz=FEATURE_ROOT/'split'/f'{item}.npz'; sidecar=FEATURE_ROOT/'split'/f'{item}.sidecar.json'
    role_status.append({'cache_id':item,'npz_exists':npz.exists(),'sidecar_exists':sidecar.exists(),'claim_allowed':False})
    assert npz.exists() and sidecar.exists(), item
(FEATURE_ROOT/'status'/'per_role_cache_status.json').write_text(json.dumps({'roles':role_status,'claim_allowed':False}, indent=2))
integrity=[]
for path in FEATURE_ROOT.rglob('*'):
    if path.is_file(): integrity.append({'path':str(path.relative_to('/kaggle/working')),'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'size':path.stat().st_size})
integrity.append({'path':'features/cifar10_r1/logs/feature_dependency_freeze.txt','sha256':hashlib.sha256(Path('/kaggle/working/feature_dependency_freeze.txt').read_bytes()).hexdigest(),'size':Path('/kaggle/working/feature_dependency_freeze.txt').stat().st_size})
(FEATURE_ROOT/'status'/'output_zip_integrity_manifest.json').write_text(json.dumps({'files':integrity,'claim_allowed':False}, indent=2))
ZIP=Path('/kaggle/working/certgen_cifar10_feature_outputs_v9_1k.zip')
if ZIP.exists(): raise FileExistsError(f'refusing to overwrite {ZIP}')
with zipfile.ZipFile(ZIP,'x',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(FEATURE_ROOT.rglob('*')):
        if path.is_file(): archive.write(path,path.relative_to('/kaggle/working'))
    archive.write('/kaggle/working/feature_dependency_freeze.txt','features/cifar10_r1/logs/feature_dependency_freeze.txt')
print('Copy back /kaggle/working/certgen_cifar10_feature_outputs_v9_1k.zip to data/kaggle_outputs/')
print('Then run commands/v9_cpu_execution/04_import_feature_zip_v9.sh')